In [3]:
import re
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
import seaborn as sns

# 1) Load your CSV
df = pd.read_csv("/home/sam/Desktop/Personal/budget-y/ANZ-7.csv",header=None)
df.columns = ["Date", "Amount", "Details"]
df["Details"] = df["Details"].astype(str)

# Extract a clean text field for modeling
def normalize(row):
    text = row["Details"]
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["text"] = df.apply(normalize, axis=1)

# 2) Weak labels
def weak_label(t):
    t = str(t).lower()

    if any(k in t for k in ["aldi", "coles", "woolworths", "iga", "foodland"]):
        return "Groceries"
    if any(k in t for k in ["upark","uber", "taxi", "lyft", "bolt", "ola", "bus", "train", "metro", "parking"]):
        return "Transport & Travel"
    if any(k in t for k in ["culinary","bottega","bakery","cafe","chatkazz", "jonny", "munooshi", "zambrero", "restaurant", "club", "bar", "uber eats", "delight", "homeboy", "grill", "pizza", "bistro", "kebab", "takeaway", "brew" ,"chaioz"]):
        return "Dining & Food"
    if any(k in t for k in ["jb","laundret","officeworks", "uniqlo", "target", "kmart", "jb hifi", "myer", "david jones", "rebel", "big w", "amazon", "ebay", "temu"]):
        return "Shopping & Retail"
    if any(k in t for k in ["united","fuel", "petrol", "bp", "caltex", "shell", "7-eleven", "ampol", "car wash"]):
        return "Auto & Fuel"
    if any(k in t for k in ["netflix", "spotify", "apple", "itunes", "subscription", "youtube", "disney", "paramount", "stan", "binge"]):
        return "Entertainment & Subscriptions"
    if any(k in t for k in ["transfer", "payment", "deposit", "atm", "withdrawal", "refund", "interest", "fee", "charge"]):
        return "Banking & Transfers"
    if any(k in t for k in ["energy", "water", "electricity", "gas", "agl", "origin", "synergy", "sa power", "telstra", "optus", "vodafone", "internet", "mobile", "nbn", "lebara"]):
        return "Utilities & Bills"
    if any(k in t for k in ["insurance", "bupa", "medibank", "nib", "allianz", "health fund", "cover","unihealth", "chemist","policy"]):
        return "Insurance & Healthcare"
    if any(k in t for k in ["university", "school", "edu", "course", "tutor", "training", "udemy", "coursera"]):
        return "Education & Learning"
    if any(k in t for k in ["charity", "donation", "ngo", "foundation", "appeal"]):
        return "Donations & Charity"
    if any(k in t for k in ["gov", "tax", "ato", "council", "license", "registration", "fine", "toll"]):
        return "Government & Fees"
    if any(k in t for k in ["pharmacy", "chemist", "medical", "clinic", "doctor", "hospital"]):
        return "Medical & Pharmacy"
    if any(k in t for k in ["hotel", "airbnb", "booking", "flight", "qantas", "jetstar", "virgin", "trip", "expedia"]):
        return "Travel & Accommodation"
    if any(k in t for k in ["gym", "fitness", "anytime fitness", "snap fitness", "pilates", "yoga", "f45"]):
        return "Health & Fitness"
    if any(k in t for k in ["hardware", "bunnings", "ikea", "home improvement", "paint", "garden", "plumbing"]):
        return "Home & Hardware"

    return "Other"

df["label"] = df["text"].apply(weak_label).fillna("Other")

# Merge rare labels into "Other"
min_per_class = 2
vc = df["label"].value_counts()
rare_labels = vc[vc < min_per_class].index
df["label"] = np.where(df["label"].isin(rare_labels), "Other", df["label"])

vc_post = df["label"].value_counts()
use_stratify = vc_post.min() >= 2

print("Label counts after merge:\n", vc_post)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    df["text"],
    df["label"],
    test_size=0.25,
    stratify=df["label"] if use_stratify else None,
    random_state=42
)

# 3) Embeddings + classifier
embedder = SentenceTransformer("all-MiniLM-L6-v2")
X_train_emb = embedder.encode(X_train.tolist(), show_progress_bar=True)
X_test_emb  = embedder.encode(X_test.tolist(), show_progress_bar=True)

clf = LogisticRegression(max_iter=200, class_weight="balanced")
clf.fit(X_train_emb, y_train)

pred = clf.predict(X_test_emb)
print("\nClassification Report:")
print(classification_report(y_test, pred))

Label counts after merge:
 label
Other                  21
Dining & Food          16
Groceries              13
Transport & Travel      8
Shopping & Retail       7
Banking & Transfers     4
Auto & Fuel             2
Name: count, dtype: int64


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Classification Report:
                     precision    recall  f1-score   support

        Auto & Fuel       0.00      0.00      0.00         1
Banking & Transfers       1.00      1.00      1.00         1
      Dining & Food       1.00      1.00      1.00         4
          Groceries       0.67      0.67      0.67         3
              Other       0.50      0.40      0.44         5
  Shopping & Retail       0.50      0.50      0.50         2
 Transport & Travel       0.50      1.00      0.67         2

           accuracy                           0.67        18
          macro avg       0.60      0.65      0.61        18
       weighted avg       0.64      0.67      0.64        18



/home/sam/miniconda3/envs/torch1070/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/sam/miniconda3/envs/torch1070/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/sam/miniconda3/envs/torch1070/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metri

In [44]:
# ==== CONFIG ====
RANDOM_STATE = 42
N_SPLITS = 3  # Reduced from 12 - more appropriate for 78 samples
N_REPEATS = 3  # Reduced from 4
USE_OVERSAMPLING = True

# ==== 2) HYBRID FEATURES & CV EVALUATION ====
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.metrics import f1_score, balanced_accuracy_score, accuracy_score, confusion_matrix
from scipy.sparse import csr_matrix, hstack
from collections import defaultdict
from imblearn.over_sampling import RandomOverSampler
import warnings

# Initialize SBERT model (reused across all folds)
sbert = SentenceTransformer("all-MiniLM-L6-v2")
print("Encoding all texts with SBERT...")
emb = sbert.encode(df['text'].to_list(), show_progress_bar=True).astype("float32")

# ==== 3) MODELS ====
models = {
    "LogReg": LogisticRegression(max_iter=200, class_weight="balanced", n_jobs=None, random_state=RANDOM_STATE),
    "RandForest": RandomForestClassifier(n_estimators=400, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1),
    "SVM_RBF": SVC(kernel="rbf", class_weight="balanced", probability=True, random_state=RANDOM_STATE),
    "MLP": MLPClassifier(hidden_layer_sizes=(256,128), activation="relu", learning_rate_init=1e-3,
                         batch_size=128, max_iter=50, random_state=RANDOM_STATE)
}

# Optional XGBoost if available
try:
    from xgboost import XGBClassifier
    models["XGBoost"] = XGBClassifier(
        n_estimators=600, max_depth=6, learning_rate=0.05, subsample=0.9, colsample_bytree=0.9,
        reg_lambda=1.0, objective="multi:softprob", tree_method="hist", eval_metric="mlogloss",
        random_state=RANDOM_STATE, n_jobs=-1
    )
except Exception as e:
    print("xgboost not available, skipping…")

# ==== 4) CV EVALUATION (NO DATA LEAKAGE) ====
rskf = RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=RANDOM_STATE)

def evaluate_model(clf, emb, texts, y):
    """
    Evaluate model using cross-validation WITHOUT data leakage.
    TF-IDF vectorizers are fit only on training data in each fold.
    """
    scores = defaultdict(list)
    labels_order = np.unique(y)
    cm_sum = np.zeros((len(labels_order), len(labels_order)), dtype=int)

    for fold_num, (train_idx, test_idx) in enumerate(rskf.split(np.zeros(len(y)), y), 1):
        # Split embeddings and texts
        emb_tr, emb_te = emb[train_idx], emb[test_idx]
        texts_tr = [texts[i] for i in train_idx]
        texts_te = [texts[i] for i in test_idx]
        ytr, yte = [y[i] for i in train_idx], [y[i] for i in test_idx]

        # FIT TF-IDF on training data only (prevents data leakage)
        tfidf_word = TfidfVectorizer(ngram_range=(1,2), min_df=1, max_df=0.9)
        tfidf_char = TfidfVectorizer(analyzer="char", ngram_range=(3,5), min_df=1)
        
        X_word_tr = tfidf_word.fit_transform(texts_tr)
        X_char_tr = tfidf_char.fit_transform(texts_tr)
        
        # TRANSFORM test data using fitted vectorizers
        X_word_te = tfidf_word.transform(texts_te)
        X_char_te = tfidf_char.transform(texts_te)

        # Combine features: [SBERT | TF-IDF word | TF-IDF char]
        Xtr = hstack([csr_matrix(emb_tr), X_word_tr, X_char_tr], format="csr")
        Xte = hstack([csr_matrix(emb_te), X_word_te, X_char_te], format="csr")

        # Oversample training data if enabled
        if USE_OVERSAMPLING:
            ros = RandomOverSampler(random_state=RANDOM_STATE)
            Xtr, ytr = ros.fit_resample(Xtr, ytr)

        # Train and predict
        clf.fit(Xtr, ytr)
        yp = clf.predict(Xte)

        # Get labels present in this fold
        test_labels = np.unique(yte)

        # Calculate metrics with proper handling of missing classes
        with warnings.catch_warnings():
            warnings.filterwarnings('ignore', message='.*y_pred contains classes not in y_true.*')
            scores["macro_f1"].append(f1_score(yte, yp, average="macro", labels=test_labels, zero_division=0))
            scores["balanced_acc"].append(balanced_accuracy_score(yte, yp))
            scores["accuracy"].append(accuracy_score(yte, yp))

        cm = confusion_matrix(yte, yp, labels=labels_order)
        cm_sum += cm

    out = {k: (np.mean(v), np.std(v)) for k, v in scores.items()}
    return out, labels_order, cm_sum

# Run evaluation for all models
print(f"\n{'='*60}")
print(f"Running {N_SPLITS}-fold CV × {N_REPEATS} repeats = {N_SPLITS*N_REPEATS} total folds")
print(f"Dataset: {len(df)} samples, {len(df['label'].unique())} classes")
print(f"Oversampling: {'Enabled' if USE_OVERSAMPLING else 'Disabled'}")
print(f"{'='*60}")

results = {}
cms = {}
texts_list = df['text'].to_list()
labels_list = df['label'].to_list()

for name, clf in models.items():
    print(f"\n=== {name} ===")
    out, order, cm = evaluate_model(clf, emb, texts_list, labels_list)
    results[name] = out
    cms[name] = (order, cm)
    print({k: f"{v[0]:.3f}±{v[1]:.3f}" for k, v in out.items()})

# ==== 5) PRINT RESULTS & CONFUSION MATRIX ====
def rank_by(metric="macro_f1"):
    return sorted(results.items(), key=lambda kv: kv[1][metric][0], reverse=True)

ranking = rank_by("macro_f1")
print(f"\n{'='*60}")
print("MODEL RANKING (by macro F1):")
print(f"{'='*60}")
for i,(name, scores) in enumerate(ranking, 1):
    print(f"{i}. {name:12s} | F1={scores['macro_f1'][0]:.3f}±{scores['macro_f1'][1]:.3f} | BalAcc={scores['balanced_acc'][0]:.3f}±{scores['balanced_acc'][1]:.3f} | Acc={scores['accuracy'][0]:.3f}±{scores['accuracy'][1]:.3f}")

# Display confusion matrix for the top model
top_name = ranking[0][0]
order, cm = cms[top_name]
print(f"\n{'='*60}")
print(f"Confusion matrix (aggregated) for best model: {top_name}")
print(f"{'='*60}")
print("Labels order:", list(order))
print(cm)

Encoding all texts with SBERT...


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

xgboost not available, skipping…

Running 3-fold CV × 3 repeats = 9 total folds
Dataset: 78 samples, 7 classes
Oversampling: Enabled

=== LogReg ===


/home/sam/miniconda3/envs/torch1070/lib/python3.11/site-packages/sklearn/model_selection/_split.py:811: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=3.
  warnings.warn(
/home/sam/miniconda3/envs/torch1070/lib/python3.11/site-packages/sklearn/model_selection/_split.py:811: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=3.
  warnings.warn(
/home/sam/miniconda3/envs/torch1070/lib/python3.11/site-packages/sklearn/model_selection/_split.py:811: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=3.
  warnings.warn(


{'macro_f1': '0.654±0.115', 'balanced_acc': '0.664±0.125', 'accuracy': '0.808±0.063'}

=== RandForest ===


...

{'macro_f1': '0.622±0.079', 'balanced_acc': '0.629±0.082', 'accuracy': '0.769±0.044'}

=== SVM_RBF ===


/home/sam/miniconda3/envs/torch1070/lib/python3.11/site-packages/sklearn/model_selection/_split.py:811: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=3.
  warnings.warn(
/home/sam/miniconda3/envs/torch1070/lib/python3.11/site-packages/sklearn/model_selection/_split.py:811: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=3.
  warnings.warn(
/home/sam/miniconda3/envs/torch1070/lib/python3.11/site-packages/sklearn/model_selection/_split.py:811: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=3.
  warnings.warn(
/home/sam/miniconda3/envs/torch1070/lib/python3.11/site-packages/sklearn/model_selection/_split.py:811: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=3.
  warnings.warn(


{'macro_f1': '0.612±0.083', 'balanced_acc': '0.609±0.083', 'accuracy': '0.761±0.047'}

=== MLP ===


/home/sam/miniconda3/envs/torch1070/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (50) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/sam/miniconda3/envs/torch1070/lib/python3.11/site-packages/sklearn/model_selection/_split.py:811: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=3.
  warnings.warn(
/home/sam/miniconda3/envs/torch1070/lib/python3.11/site-packages/sklearn/model_selection/_split.py:811: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=3.
  warnings.warn(


{'macro_f1': '0.614±0.092', 'balanced_acc': '0.633±0.091', 'accuracy': '0.774±0.053'}

MODEL RANKING (by macro F1):
1. LogReg       | F1=0.654±0.115 | BalAcc=0.664±0.125 | Acc=0.808±0.063
2. RandForest   | F1=0.622±0.079 | BalAcc=0.629±0.082 | Acc=0.769±0.044
3. MLP          | F1=0.614±0.092 | BalAcc=0.633±0.091 | Acc=0.774±0.053
4. SVM_RBF      | F1=0.612±0.083 | BalAcc=0.609±0.083 | Acc=0.761±0.047

Confusion matrix (aggregated) for best model: LogReg
Labels order: [np.str_('Banking & Transfers'), np.str_('Dining & Food'), np.str_('Groceries'), np.str_('Insurance & Healthcare'), np.str_('Other'), np.str_('Shopping & Retail'), np.str_('Transport & Travel')]
[[ 6  0  0  0  0  0  0]
 [ 0 85  2  0  0  0  0]
 [ 0  0 60  0  0  0  0]
 [ 0  3  0  0  3  0  0]
 [ 0 12  3  0 15  2  1]
 [ 0  2  3  0  4 15  0]
 [ 0  2  0  0  8  0  8]]


In [46]:
from sentence_transformers import SentenceTransformer, util
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")

# Compute class centroids from existing labeled data
class_embeddings = {}
for label in df["label"].unique():
    emb = model.encode(df.loc[df["label"] == label, "text"].tolist(), normalize_embeddings=True)
    class_embeddings[label] = np.mean(emb, axis=0)

# Predict for a new unseen transaction
new_txn = "CHARLESWORTH NUTS PT ADELAIDE"
new_emb = model.encode(new_txn, normalize_embeddings=True)

# Compute cosine similarities
scores = {label: util.cos_sim(new_emb, centroid)[0][0].item() for label, centroid in class_embeddings.items()}
pred_label = max(scores, key=scores.get)
print(pred_label, scores)


Other {'Dining & Food': 0.5137254595756531, 'Groceries': 0.3650323748588562, 'Transport & Travel': 0.5127879977226257, 'Shopping & Retail': 0.5469932556152344, 'Other': 0.6894878149032593, 'Insurance & Healthcare': 0.48687756061553955, 'Banking & Transfers': 0.16050444543361664}


In [47]:
# pip install sentence-transformers scikit-learn scipy
import numpy as np, pandas as pd, re
from scipy.sparse import csr_matrix, hstack, vstack
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

# --- 1) Data ---
texts = df["text"].astype(str).tolist()
labels = df["label"].astype(str).tolist()

X_train_text, X_test_text, y_train, y_test = train_test_split(
    texts, labels, test_size=0.25, stratify=labels, random_state=42
)

# --- 2) Embeddings ---
sbert = SentenceTransformer("all-MiniLM-L6-v2")
X_train_emb = sbert.encode(X_train_text, normalize_embeddings=True).astype("float32")
X_test_emb  = sbert.encode(X_test_text,  normalize_embeddings=True).astype("float32")

scaler = StandardScaler(with_mean=False)
X_train_emb_s = scaler.fit_transform(csr_matrix(X_train_emb))
X_test_emb_s  = scaler.transform(csr_matrix(X_test_emb))

# --- 3) TF-IDF blocks ---
tfidf_word = TfidfVectorizer(ngram_range=(1,2), min_df=2, max_df=0.95)
tfidf_char = TfidfVectorizer(analyzer="char", ngram_range=(3,5), min_df=2)

X_train_w = tfidf_word.fit_transform(X_train_text)
X_test_w  = tfidf_word.transform(X_test_text)
X_train_c = tfidf_char.fit_transform(X_train_text)
X_test_c  = tfidf_char.transform(X_test_text)

# --- 4) Early fusion features ---
X_train = hstack([X_train_emb_s, X_train_w, X_train_c], format="csr")
X_test  = hstack([X_test_emb_s,  X_test_w,  X_test_c],  format="csr")

# --- 5) Train classifier ---
clf = LogisticRegression(max_iter=300, class_weight="balanced", random_state=42)
clf.fit(X_train, y_train)

p_ml = clf.predict_proba(X_test)
classes = list(clf.classes_)

# --- 6) Compute n-gram (lexical) centroids per class ---
def build_centroids(Xw, Xc, y):
    cents = {}
    for cls in classes:
        idx = [i for i, yy in enumerate(y) if yy == cls]
        if not idx:
            cents[cls] = csr_matrix((1, Xw.shape[1] + Xc.shape[1]))
            continue
        Cw, Cc = Xw[idx], Xc[idx]
        cents[cls] = csr_matrix(np.hstack([Cw.mean(axis=0), Cc.mean(axis=0)]))
    return cents

centroids = build_centroids(X_train_w, X_train_c, y_train)

def ngram_scores(texts):
    Xw, Xc = tfidf_word.transform(texts), tfidf_char.transform(texts)
    Xcomb = csr_matrix(np.hstack([Xw.toarray(), Xc.toarray()]))
    scores = np.zeros((Xcomb.shape[0], len(classes)))
    for j, cls in enumerate(classes):
        sims = cosine_similarity(Xcomb, centroids[cls])
        scores[:, j] = sims.ravel()
    scores = np.maximum(scores, 0)
    row_sum = scores.sum(axis=1, keepdims=True)
    scores = scores / np.maximum(row_sum, 1e-8)
    return scores

p_lex = ngram_scores(X_test_text)

# --- 7) Late fusion of ML and lexical similarity ---
alpha = 0.8  # weight for ML model
p_fused = alpha * p_ml + (1 - alpha) * p_lex
p_fused /= p_fused.sum(axis=1, keepdims=True)
y_pred = [classes[i] for i in np.argmax(p_fused, axis=1)]

# --- 8) Evaluation ---
print(classification_report(y_test, y_pred, digits=3))


                     precision    recall  f1-score   support

Banking & Transfers      1.000     1.000     1.000         1
      Dining & Food      0.875     1.000     0.933         7
          Groceries      1.000     0.800     0.889         5
              Other      0.667     0.667     0.667         3
  Shopping & Retail      1.000     1.000     1.000         2
 Transport & Travel      1.000     1.000     1.000         2

           accuracy                          0.900        20
          macro avg      0.924     0.911     0.915        20
       weighted avg      0.906     0.900     0.899        20



In [52]:
# --- 9) Create a dataframe with results ---
results_df = pd.DataFrame({
    "Transaction": X_test_text,
    "True_Label": y_test,
    "Predicted_Label": y_pred
})

# --- 10) Filter only incorrect predictions ---
incorrect_df = results_df[results_df["True_Label"] != results_df["Predicted_Label"]]

# Show them sorted by true label (optional)
incorrect_df = incorrect_df.sort_values(by="True_Label").reset_index(drop=True)

# Print clearly
pd.set_option("display.max_colwidth", None)
print("Incorrect Predictions:")
print(incorrect_df)



Incorrect Predictions:
                          Transaction True_Label Predicted_Label
0  WOOLWORTHS/7 GRASSMERE RD PROSPECT  Groceries           Other
1       ASIAN GOURMET RST PL ADELAIDE      Other   Dining & Food


In [6]:
# pip install scikit-learn pandas numpy

import re
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# --- 1) Define your categories and keywords ---
category_keywords = {
    "Groceries": ["aldi", "coles", "woolworths", "iga", "foodland"],
    "Transport & Travel": ["upark","uber", "taxi", "lyft", "bolt", "ola", "bus", "train", "metro", "parking"],
    "Dining & Food": ["culinary","bottega","bakery","cafe","chatkazz", "jonny", "munooshi", "zambrero", "restaurant", "club", "bar", "uber eats", "delight", "homeboy", "grill", "pizza", "bistro", "kebab", "takeaway", "brew","chaioz"],
    "Shopping & Retail": ["jb","laundret","officeworks", "uniqlo", "target", "kmart", "jb hifi", "myer", "david jones", "rebel", "big w", "amazon", "ebay", "temu"],
    "Auto & Fuel": ["united","fuel", "petrol", "bp", "caltex", "shell", "7-eleven", "ampol", "car wash"],
    "Entertainment & Subscriptions": ["netflix", "spotify", "apple", "itunes", "subscription", "youtube", "disney", "paramount", "stan", "binge"],
    "Banking & Transfers": ["transfer", "payment", "deposit", "atm", "withdrawal", "refund", "interest", "fee", "charge"],
    "Utilities & Bills": ["energy", "water", "electricity", "gas", "agl", "origin", "synergy", "sa power", "telstra", "optus", "vodafone", "internet", "mobile", "nbn", "lebara"],
    "Insurance & Healthcare": ["insurance", "bupa", "medibank", "nib", "allianz", "health fund", "cover", "unihealth", "chemist", "policy"],
    "Education & Learning": ["university", "school", "edu", "course", "tutor", "training", "udemy", "coursera"],
    "Donations & Charity": ["charity", "donation", "ngo", "foundation", "appeal"],
    "Government & Fees": ["gov", "tax", "ato", "council", "license", "registration", "fine", "toll"],
    "Medical & Pharmacy": ["pharmacy", "chemist", "medical", "clinic", "doctor", "hospital"],
    "Travel & Accommodation": ["hotel", "airbnb", "booking", "flight", "qantas", "jetstar", "virgin", "trip", "expedia"],
    "Health & Fitness": ["gym", "fitness", "anytime fitness", "snap fitness", "pilates", "yoga", "f45"],
    "Home & Hardware": ["hardware", "bunnings", "ikea", "home improvement", "paint", "garden", "plumbing"],
    "Other": []
}

# --- 2) Prepare category corpus ---
category_texts = [" ".join(words) for words in category_keywords.values()]
category_names = list(category_keywords.keys())

# --- 3) Prepare your transaction text ---
df["Details"] = df["Details"].astype(str).str.lower()
df["text"] = df["Details"].apply(lambda x: re.sub(r"[^a-z0-9 ]", " ", x))

# --- 4) TF-IDF vectorization ---
vectorizer = TfidfVectorizer(ngram_range=(1, 2), stop_words="english")
all_texts = category_texts + df["text"].tolist()
X = vectorizer.fit_transform(all_texts)

# Split into category and transaction matrices
X_cat = X[:len(category_names)]
X_txn = X[len(category_names):]

# --- 5) Compute cosine similarity ---
sims = cosine_similarity(X_txn, X_cat)
best_idx = sims.argmax(axis=1)
df["Predicted_Label"] = [category_names[i] for i in best_idx]
df["Similarity_Score"] = sims.max(axis=1)

# --- 6) Save results ---
output_path = "/home/sam/Desktop/Personal/budget-y/transaction_cosine_categories.csv"
df.to_csv(output_path, index=False)

print(f"✅ Saved categorized transactions to:\n{output_path}")
print(df[["Details", "Predicted_Label", "Similarity_Score"]].head(20))


✅ Saved categorized transactions to:
/home/sam/Desktop/Personal/budget-y/transaction_cosine_categories.csv
                                    Details    Predicted_Label  \
0      officeworks 0503          nailsworth  Shopping & Retail   
1      chatkazz lightsview       lightsview      Dining & Food   
2      chatkazz lightsview       lightsview      Dining & Food   
3        united richmond           richmond        Auto & Fuel   
4   bakery on o'connell-      north adelaid      Dining & Food   
5        jb hi fi adelaide ci      adelaide  Shopping & Retail   
6    culinary escapades pty lt campbelltown      Dining & Food   
7        zambrero sa pty ltd       adelaide  Utilities & Bills   
8        aldi stores               prospect          Groceries   
9   sq *chaioz                north adelaid      Dining & Food   
10     coles 4948                greenacres          Groceries   
11       rundle mall foodland      adelaide          Groceries   
12       retailors ops pty ltd     